# Segment 4 — Retrieval (RAG)

Answer questions using **your own documents**:
find the relevant docs, paste them into the prompt, and ask the model to answer from them.

## Setup
Run these two cells first: install the libraries, then create the `llm`.

In [ ]:
# Run once. Installs the workshop libraries.
%pip install -q langchain langchain-core langchain-openai langgraph pydantic

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI

# Paste your OpenRouter key when prompted (get one at https://openrouter.ai/keys).
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

# OpenRouter uses the OpenAI API, so we use ChatOpenAI and just change the URL.
# Pass the key explicitly: ChatOpenAI does NOT read OPENROUTER_API_KEY on its own.
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",                       # any model from openrouter.ai/models
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)
print("LLM ready:", llm.model_name)

## A tiny knowledge base + a simple search

In [ ]:
documents = [
    "Refunds are processed within 5-7 business days to the original payment method.",
    "Standard shipping takes 3-5 business days. Express shipping takes 1-2 days.",
    "The Pro plan costs $20/month and includes unlimited projects and SSO.",
    "To reset your password, go to Settings > Security and click 'Reset password'.",
]

# Keep documents that share words with the question.
def find_relevant(question):
    q_words = set(question.lower().split())
    return [doc for doc in documents if q_words & set(doc.lower().split())]

## 1. Retrieve

In [ ]:
question = "How long do refunds take?"
context = "\n".join(find_relevant(question))
print(context)

## 2. Answer using only that context

In [ ]:
answer = llm.invoke(
    f"Answer using ONLY this context.\n\nContext:\n{context}\n\nQuestion: {question}"
)
print(answer.content)